# Train XGBoost for masked vertebral height only

This notebook predicts one value: the mean frontal height of a masked vertebra. Mean height is (left height + right height) / 2.

**Research use only.** This reconstructs observed height from surrounding landmarks; it is not a clinically validated estimate of healthy anatomy.

## Flow

Four surrounding vertebrae → calculate normalized context morphology → interpolate the target height from the nearest pair → one XGBoost model predicts a height residual → final normalized target height → held-out evaluation.

In [ ]:
# If needed, uncomment and run once:
# %pip install -r ../../requirement.txt

## 1. Configuration

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBRegressor


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "dataset").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.train_masked_height_xgboost import (
    height_metrics,
    height_targets,
    prediction_frame,
)
from src.train_masked_morphology_xgboost import load_dataset

DATASET_ROOT = REPO_ROOT / "dataset/processed/masked_morphology_coco_nih_lumos"
OUTPUT_DIR = REPO_ROOT / "outputs/masked_morphology/xgboost_height_only_notebook_v1"
SEED = 20260814
LIMIT_PER_SPLIT = None  # Use 200 for a quick trial.
SAVE_ARTIFACTS = True

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

## 2. Load the fixed patient/case-grouped splits

The existing dataset loader verifies required columns, finite values, positive weights, unique sample IDs, and no group overlap between train, validation, and test.

In [ ]:
frames, schema, feature_columns = load_dataset(DATASET_ROOT, limit_per_split=LIMIT_PER_SPLIT)
split_summary = pd.DataFrame({
    split: {
        "samples": len(frame),
        "groups": frame["group_id"].nunique(),
        "images": frame["image_path"].nunique(),
    }
    for split, frame in frames.items()
}).T
display(split_summary)

## 3. Derive one height target

The output is height only. The model may use all 28 normalized morphology measurements from the four surrounding vertebrae because their shape and spacing can help estimate height; target position and identifiers are excluded.

The target is trained as a correction: target residual = actual normalized height − interpolated normalized height.

In [ ]:
X = {split: frame.loc[:, feature_columns] for split, frame in frames.items()}
targets = {split: height_targets(frame) for split, frame in frames.items()}

print(f"The single-output height model receives {len(feature_columns)} context features:")
print(feature_columns)
display(pd.concat([X["train"].head(5), targets["train"].head(5)], axis=1).round(4))

In [ ]:
assert list(X["train"].columns) == feature_columns
assert all(np.isfinite(table.to_numpy()).all() for table in X.values())
assert all(np.isfinite(table.to_numpy()).all() for table in targets.values())
assert not any(name.startswith(("y_", "baseline_")) for name in feature_columns)

group_sets = {split: set(frame["group_id"]) for split, frame in frames.items()}
assert group_sets["train"].isdisjoint(group_sets["val"])
assert group_sets["train"].isdisjoint(group_sets["test"])
assert group_sets["val"].isdisjoint(group_sets["test"])
print("Audit passed: context-only inputs, one height target, and no group leakage.")

## 4. Evaluate interpolation before training

The baseline is the average mean height of the nearest superior and inferior vertebrae. A useful learned model must clearly improve on it.

In [ ]:
baseline_rows = []
for split in ("val", "test"):
    table = targets[split]
    baseline_rows.append({
        "split": split,
        **height_metrics(
            table["actual_mean_height_norm"].to_numpy(),
            table["baseline_mean_height_norm"].to_numpy(),
            table["baseline_mean_height_norm"].to_numpy(),
            frames[split]["sample_weight"].to_numpy(),
        ),
    })
baseline_metrics = pd.DataFrame(baseline_rows)
display(baseline_metrics[[
    "split", "baseline_mae", "baseline_rmse", "baseline_r2",
    "baseline_within_5pct_reference_percent",
    "baseline_within_10pct_reference_percent",
]].round(4))

## 5. Train one XGBoost regressor

In [ ]:
MODEL_PARAMS = {
    "objective": "reg:pseudohubererror",
    "eval_metric": "mae",
    "n_estimators": 1600,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 4.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 2.0,
    "tree_method": "hist",
    "device": "cpu",
    "early_stopping_rounds": 80,
    "random_state": SEED,
    "n_jobs": 4,
}
display(pd.Series(MODEL_PARAMS, name="value").to_frame())

In [ ]:
model = XGBRegressor(**MODEL_PARAMS)
model.fit(
    X["train"],
    targets["train"]["mean_height_residual"],
    sample_weight=frames["train"]["sample_weight"].to_numpy(),
    eval_set=[(X["val"], targets["val"]["mean_height_residual"])],
    sample_weight_eval_set=[frames["val"]["sample_weight"].to_numpy()],
    verbose=False,
)
history = model.evals_result()
print("Best iteration:", model.best_iteration)

In [ ]:
values = history["validation_0"]["mae"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(values, color="#219ebc")
ax.axvline(model.best_iteration, color="#fb8500", linestyle="--")
ax.set(title="Validation MAE", xlabel="Boosting iteration", ylabel="Residual MAE")
plt.show()

## 6. Held-out evaluation

Within 5% means the predicted height differs by no more than 0.05 times the median height of the four context vertebrae. The test split is inspected only after training.

In [ ]:
predicted_residual = {
    split: model.predict(X[split]) for split in ("val", "test")
}
predicted_height = {
    split: targets[split]["baseline_mean_height_norm"].to_numpy() + predicted_residual[split]
    for split in ("val", "test")
}

metric_rows = []
prediction_tables = []
for split in ("val", "test"):
    table = targets[split]
    metric_rows.append({
        "split": split,
        **height_metrics(
            table["actual_mean_height_norm"].to_numpy(),
            table["baseline_mean_height_norm"].to_numpy(),
            predicted_height[split],
            frames[split]["sample_weight"].to_numpy(),
        ),
    })
    prediction_tables.append(prediction_frame(
        frames[split], table, predicted_residual[split], split=split
    ))

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_tables, ignore_index=True)
display(metrics[[
    "split", "baseline_mae", "model_mae", "baseline_rmse", "model_rmse",
    "baseline_within_5pct_reference_percent", "model_within_5pct_reference_percent",
    "baseline_within_10pct_reference_percent", "model_within_10pct_reference_percent",
    "baseline_r2", "model_r2", "relative_mae_improvement_percent",
]].round(4))

In [ ]:
test_metrics = metrics.loc[metrics["split"].eq("test")].iloc[0]
labels = ["Within 5%", "Within 10%"]
baseline_values = [
    test_metrics["baseline_within_5pct_reference_percent"],
    test_metrics["baseline_within_10pct_reference_percent"],
]
model_values = [
    test_metrics["model_within_5pct_reference_percent"],
    test_metrics["model_within_10pct_reference_percent"],
]
positions = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(positions - 0.18, baseline_values, 0.36, label="Interpolation", color="#9aa0a6")
ax.bar(positions + 0.18, model_values, 0.36, label="XGBoost", color="#219ebc")
ax.set_xticks(positions, labels)
ax.set_ylabel("Weighted test accuracy (%)")
ax.set_ylim(0, 100)
ax.legend()
plt.show()

In [ ]:
test_predictions = predictions.loc[predictions["split"].eq("test")]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    test_predictions["actual_mean_height_norm"],
    test_predictions["predicted_mean_height_norm"],
    s=10, alpha=0.3, color="#219ebc",
)
limits = [
    min(test_predictions["actual_mean_height_norm"].min(), test_predictions["predicted_mean_height_norm"].min()),
    max(test_predictions["actual_mean_height_norm"].max(), test_predictions["predicted_mean_height_norm"].max()),
]
ax.plot(limits, limits, "--", color="black")
ax.set(xlabel="Actual normalized mean height", ylabel="Predicted normalized mean height")
plt.show()

## 7. Inspect feature importance and largest errors

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
display(feature_importance)

largest_errors = test_predictions.nlargest(20, "absolute_error_norm")
display(largest_errors[[
    "sample_id", "source_dataset", "image_path", "target_chain_rank",
    "actual_mean_height_norm", "baseline_mean_height_norm",
    "predicted_mean_height_norm", "absolute_error_norm",
]])

## 8. Save and verify the height-only artifact

In [ ]:
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_model(OUTPUT_DIR / "mean_height.json")
    metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
    with (OUTPUT_DIR / "training_history.json").open("w", encoding="utf-8") as file:
        json.dump(history, file, indent=2)
        file.write("\n")
    manifest = {
        "task": "four-context masked mean vertebral height residual regression",
        "research_use_only": True,
        "dataset_root": str(DATASET_ROOT),
        "feature_columns": feature_columns,
        "target_formula": "0.5 * (left_height_norm + right_height_norm)",
        "normalization": "median mean height of the four context vertebrae",
        "best_iteration": int(model.best_iteration),
        "model_parameters": MODEL_PARAMS,
    }
    with (OUTPUT_DIR / "run_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
        file.write("\n")

    reloaded = XGBRegressor()
    reloaded.load_model(OUTPUT_DIR / "mean_height.json")
    assert np.isfinite(reloaded.predict(X["test"].head(10))).all()
    print("Saved and reloaded:", OUTPUT_DIR)

## Interpretation rule

Keep XGBoost only if it clearly improves held-out MAE and 5%/10% accuracy over interpolation. Otherwise, use interpolation because it is simpler and easier to audit.